<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/mean_rainfall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --pre xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(
    project ='ee-grmntfrancis0',
    opt_url = 'https://earthengine-highvolume.googleapis.com'
)

In [ ]:
import geemap

In [ ]:
map = geemap.Map(basemap = 'SATELLITE')
map

In [ ]:
point = map.draw_last_feature.geometry()
point

In [ ]:
country = (
    ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM2")
    .filterBounds(point)
    .geometry().simplify(1000)
)

In [ ]:
map.addLayer(country,{},'South Sudan')

In [ ]:
pr = (
    ee.ImageCollection("UCSB-CHC/CHIRPS/V3/DAILY_RNL")
    .filterDate('1985','2025')
    .select(['precipitation'],['pr'])
)

pr

In [ ]:
from shapely.geometry import shape

country_shape = shape(country.getInfo())
country_shape

In [ ]:
from xee import helpers

In [ ]:
grid_params = helpers.fit_geometry(
    geometry = country_shape,
    grid_crs = 'EPSG:4326',
    grid_scale = (0.3, -0.3)
)


In [ ]:
import xarray as xr

In [ ]:
ds = xr.open_dataset(
    pr,
    engine = 'ee',
    **grid_params
)
ds

In [ ]:
ds = ds.sortby('time') * 1

In [ ]:
!pip install rioxarray

In [ ]:
import rioxarray

In [ ]:
country_gdf = geemap.ee_to_gdf(ee.FeatureCollection(country))

country_gdf.plot()

In [ ]:
ds.isel(time = 0).pr.plot(x = 'x', y = 'y', robust =True)

In [ ]:
ds_clip = ds.rio.clip(country_gdf.geometry, country_gdf.crs)

In [ ]:
ds_clip.isel(time = 0).pr.plot(x = 'x', y ='y', robust = True)

In [ ]:
pr_annual = ds_clip.resample(time = 'YE').sum('time')

In [ ]:
import numpy as np

In [ ]:
pr_anomaly = pr_annual - pr_annual.mean(dim = 'time')
pr_anomaly = pr_anomaly.where(pr_anomaly != 0, np.nan)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
pr_anomaly.pr.plot.contourf(
    x = 'x', y = 'y', robust = True, col_wrap = 6,
    cmap = 'turbo_r', col = 'time', levels = 10, size=4, aspect=1.6, fontsize = 20
)
plt.show()

In [ ]:
pr_monthly = ds_clip.resample(time = 'M').sum('time')

In [ ]:
pr_month = pr_monthly.groupby('time.month').mean('time')

In [ ]:
pr_month = pr_month.where(pr_month != 0, np.nan)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
pr_month.pr.plot.contourf(
    x = 'x', y = 'y', robust = True, col = 'month', col_wrap = 6,
    cmap = 'turbo_r', levels = 15, cbar_kwargs = {'orientation': 'vertical', 'label': 'Precipitation scale in mm/year'}, size=4, aspect=1.6
)
plt.show()

In [ ]:
pr_flood = pr_monthly.sel(time = '2020')
pr_flood = pr_flood.where(pr_flood != 0, np.nan)

In [ ]:
pr_flood.pr.plot.contourf(
    x = 'x', y = 'y', robust = True, col = 'time', col_wrap = 6,
    cmap = 'turbo_r', levels = 12, cbar_kwargs = {'orientation': 'vertical', 'label': 'Precipitation scale in mm/year'}, size=4, aspect=1.5
)
plt.show()

In [ ]:
ds_clip.sel(time = '2020-02').pr.plot.contourf(
    x = 'x', y = 'y', robust =True, cmap = 'turbo_r', levels = 12,
    col = 'time', col_wrap = 7, cbar_kwargs = {'orientation': 'vertical', 'label': 'Precipitation scale in mm/year'}, size=4, aspect=1.6
)
plt.show()

In [ ]:
mean_annual_pr = pr_annual.mean(dim=['x', 'y'])

plt.figure(figsize=(12, 6))
mean_annual_pr.pr.plot(label='Annual Precipitation', linewidth=1.3)

# Add a smoothing line (e.g., 5-year rolling mean)
mean_annual_pr.pr.rolling(time=5, center=True).mean().plot(color='red', linestyle='--', label='5-Year Rolling Mean', linewidth=2)

plt.title('SOUTH SUDAN: ANNUAL MEAN RAINFALL (1985-2025)')
plt.xlabel('Year')
plt.ylabel('Mean Annual Precipitation (mm/year)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('mean_annual rainfall', dpi = 360, bbox_inches = 'tight')
plt.show()

In [ ]:
ds_clip.mean(dim = 'time').pr.plot.contourf(x = 'x', y ='y', figsize=(8, 6), aspect=1.6)

plt.tight_layout(rect=[0.02, 0.05, 0.98, 0.95])
plt.show()